### Simplified Example 10 of the PHT3D Manual
Toluene enters from the left into a homogeous aquifer with parallel flow from left to right.
It degrades via 1st order decay consuming various electron acceptors. The model simulates the corresponding terminal electron acceptor processes and some secondary reactions 

In [1]:
import flopy
import numpy as np

In [2]:
length_units = 'meters'
time_units = 'days'
sim_name = 'ex10_simple' # Note that the name should only contain small letters
gwf_name = f'gwf_{sim_name}'
gwt_name = f'gwt_{sim_name}'
sim_ws = './mf6'
concentration_name = 'concentration' # name for concentration
rtmf6_sol_number_name = 'rtmf6_sol_number' # solution number of PHREEQC solution

In [3]:
sim = flopy.mf6.MFSimulation(sim_name=sim_name, sim_ws=sim_ws)

nper = 1  
perlen = 500 
nstp = 100

In [4]:
tdis = flopy.mf6.ModflowTdis(
    sim, 
    nper=nper, 
    perioddata=[(perlen, nstp, 1.0)], 
    time_units=time_units)

#### Groundwater flow (GWF)

In [5]:
gwf = flopy.mf6.ModflowGwf(
    sim,
    modelname=gwf_name,
    save_flows=True,
    model_nam_file=f"{gwf_name}.nam",
)

In [6]:
#%% Flow solver parameters
nouter, ninner = 300, 600
hclose, rclose, relax = 1e-6, 1e-6, 1.0


imsgwf = flopy.mf6.ModflowIms(
    sim,
    complexity="complex",
    print_option="SUMMARY",
    outer_dvclose=hclose,
    outer_maximum=nouter,
    under_relaxation="NONE",
    inner_maximum=ninner,
    inner_dvclose=hclose,
    rcloserecord=rclose,
    linear_acceleration="CG",
    scaling_method="NONE",
    reordering_method="NONE",
    relaxation_factor=relax,
    filename=f"{gwf_name}.ims",
)
sim.register_ims_package(imsgwf, [gwf.name])

In [7]:
nlay = 1  # Number of layers
ncol = 80 # Number of columns
nrow = 40  # Number of rows
delr = 2.5 # 
delc = 1.25
top = 10
botm = np.array([0])
dis = flopy.mf6.ModflowGwfdis(
    gwf,
    length_units=length_units,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top = top,
    botm = botm,
    filename=f"{gwf_name}.dis",
    nogrb=True,
)

In [8]:
k11 = 10.0  # Horizontal hydraulic conductivity ($m/d$)
k33 = k11  # Vertical hydraulic conductivity ($m/d$)
icelltype = 0 # saturated thickness is constant --> confined case


npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    save_flows=True,
    save_saturation=True,
    icelltype=icelltype,
    k=k11,
    k33=k33,
    save_specific_discharge=True,
    filename=f"{gwf_name}.npf",
)

In [9]:
flopy.mf6.ModflowGwfic(gwf, strt=1, filename=f"{gwf_name}.ic")

package_name = ic
filename = gwf_ex10_simple.ic
package_type = ic
model_or_simulation_package = model
model_name = gwf_ex10_simple

Block griddata
--------------------
strt
{constant 1}



In [10]:
#%% CHD
headup = 5
headdwn = 3
chd_spd = []

sourcec = np.zeros(nrow) # SOLUTION 0 in advect.pqi will enter as background via the CHD boundary

# Toluene containing in SOLUTION 1 in advect.pqi is the source zone of the contaminant plume
# in the middle stretch of the inflow boundary:
sourcec[13:25] = 1.0 

auxiliary = [
    concentration_name, # name for concentration
    rtmf6_sol_number_name # solution number of PHREEQC solution
]

for i in range(nrow):
  chd_spd.append([(0, i, 0), headup,sourcec[i],sourcec[i]])
  chd_spd.append([(0, i, ncol-1), headdwn,sourcec[i],sourcec[i]])

chd = flopy.mf6.ModflowGwfchd(
    gwf,
    maxbound=len(chd_spd),
    stress_period_data=chd_spd,
    save_flows=False,
    auxiliary=auxiliary,
    pname="CHD",
    filename=f"{gwf_name}.chd",
)

In [11]:
oc_gwf = flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord=f"{gwf_name}.hds",
    budget_filerecord=f"{gwf_name}.cbb",
    headprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 6, "GENERAL")],
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    printrecord=[("HEAD", "LAST"), ("BUDGET", "LAST")],
)

#### Transport (GWT)

In [12]:
gwt = flopy.mf6.MFModel(
    sim,
    model_type="gwt6",
    modelname=gwt_name,
    model_nam_file=f"{gwt_name}.nam"
)

In [13]:
#%% Transport solver parameters
imsgwt = flopy.mf6.ModflowIms(
    sim,
    print_option="SUMMARY",
    outer_dvclose=hclose,
    outer_maximum=nouter,
    under_relaxation="NONE",
    inner_maximum=ninner,
    inner_dvclose=hclose,
    rcloserecord=rclose,
    linear_acceleration="BICGSTAB",
    scaling_method="NONE",
    reordering_method="NONE",
    relaxation_factor=relax,
    filename=f"{gwt_name}.ims",
)
sim.register_ims_package(imsgwt, [gwt.name])

In [14]:
gwt_dis = flopy.mf6.ModflowGwtdis(
    gwt,
    length_units=length_units,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top = top,
    botm = botm,
    filename=f"{gwt_name}.dis",
    nogrb=True,
)

In [15]:
#%% Transport IC
gwt_ic = flopy.mf6.ModflowGwtic(
    gwt, 
    strt=0,  # rtmf6 solution number
    filename=f"{gwt_name}.ic")

In [16]:
#%% Transport SSM
sourcerecarray = ['CHD', 'aux', concentration_name]

ssm = flopy.mf6.ModflowGwtssm(
    gwt, 
    sources=sourcerecarray, 
    save_flows=True,
    print_flows=True,
    filename=f"{gwt_name}.ssm"
)

In [17]:
adv = flopy.mf6.ModflowGwtadv(
    gwt,
    scheme="UPSTREAM",
)

In [18]:
dispersivity = 0.5
transverse_horizontal_dispersivity = dispersivity * 0.2
transverse_vertical_dispersivity = dispersivity * 0.1

dsp = flopy.mf6.ModflowGwtdsp(
            gwt,
            xt3d_off=True,
            alh=dispersivity,
            ath1=transverse_horizontal_dispersivity,
            atv=transverse_vertical_dispersivity, 
            filename=f"{gwt_name}.dsp",
        )

In [19]:
first_order_decay = None
porosity = 0.3

mst = flopy.mf6.ModflowGwtmst(
    gwt,
    porosity=porosity,
    first_order_decay=first_order_decay,
    filename=f"{gwt_name}.mst",
)

In [20]:
oc_gwt = flopy.mf6.ModflowGwtoc(
    gwt,
    budget_filerecord=f"{gwt_name}.cbb",
    concentration_filerecord=f"{gwt_name}.ucn",
    concentrationprintrecord=[
        ("COLUMNS", 10, "WIDTH", 15, "DIGITS", 10, "GENERAL")
    ],
    saverecord=[("CONCENTRATION", "ALL"), 
                ("BUDGET", "ALL")
                ],
    printrecord=[("CONCENTRATION", "ALL"), 
                 ("BUDGET", "ALL")
                    ],
)

In [21]:
#%% GWF-GWT Exchange
flopy.mf6.ModflowGwfgwt(
    sim,
    exgtype="GWF6-GWT6",
    exgmnamea=gwf_name,
    exgmnameb=gwt_name,
    filename=f"{sim_name}.gwfgwt",
)

package_name = ex10_simple.gwfgwt
filename = ex10_simple.gwfgwt
package_type = gwfgwt
model_or_simulation_package = simulation
simulation_name = ex10_simple


In [22]:
# writing the MF6 files to be used as templates for the RTMF6 input, such as initial conditions 
sim.write_simulation()

writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_-1...
  writing solution package ims_0...
  writing package ex10_simple.gwfgwt...
  writing model gwf_ex10_simple...
    writing model name file...
    writing package dis...
    writing package npf...
    writing package ic...
    writing package chd...
    writing package oc...
  writing model gwt_ex10_simple...
    writing model name file...
    writing package dis...
    writing package ic...
    writing package ssm...
    writing package adv...
    writing package dsp...
    writing package mst...
    writing package oc...
